# 07 — Loading the official Qwen3 weights into our model (M2.1)

Our `Qwen3ForCausalLM` was verified layer by layer against Hugging Face with
*random* weights (M1). This notebook loads **trained** weights from the official
checkpoint `Qwen/Qwen3-0.6B-Base` into that same code.

**Two modes**

* default (CI, no download): a *synthetic* checkpoint with random weights in the exact
  official file layout is generated in a temporary directory, so every cell runs in
  seconds on a CPU;
* real weights: set `QWEN3_WEIGHTS_DIR=<checkpoint dir>` (see
  `scripts/m2_local_run.sh`) **and** `QWEN3_NB_PRETRAINED=1` before starting Jupyter.
  Needs ~4 GiB of free RAM in float32.

Write-up: `docs/components/pretrained.md`. Parity script: `scripts/verify_pretrained.py`.

In [ ]:
import json
import os
import struct
import tempfile
from pathlib import Path

import torch

from qwen3_scratch.config import QWEN3_0_6B_BASE, QWEN3_TINY
from qwen3_scratch.generation import greedy_generate
from qwen3_scratch.pretrained import QWEN3_0_6B_BASE_REVISION, load_pretrained
from qwen3_scratch.synthetic import write_synthetic_checkpoint
from qwen3_scratch.tokenizer import Qwen3Tokenizer
from qwen3_scratch.weight_loader import SafetensorsCheckpoint, build_name_map, expected_shapes

REAL = os.environ.get("QWEN3_NB_PRETRAINED") == "1" and bool(os.environ.get("QWEN3_WEIGHTS_DIR"))
if REAL:
    ckpt_dir = Path(os.environ["QWEN3_WEIGHTS_DIR"])
else:
    import dataclasses

    small = dataclasses.replace(QWEN3_TINY, vocab_size=512, max_position_embeddings=256)
    ckpt_dir = write_synthetic_checkpoint(small, tempfile.mkdtemp(), seed=0, weight_std=0.1)
print("mode:", "REAL Qwen3-0.6B-Base" if REAL else "synthetic RANDOM weights (official layout)")
print("files:", sorted(p.name for p in ckpt_dir.iterdir()))
print("pinned revision used in this project:", QWEN3_0_6B_BASE_REVISION)

## 1. How the weights are stored: Safetensors

A `.safetensors` file is deliberately simple:

```
[8 bytes: header length N, little-endian uint64]
[N bytes: JSON header  {"name": {"dtype": "BF16", "shape": [...], "data_offsets": [a, b]}, ...}]
[raw tensor bytes, back to back]
```

There is no pickled Python inside, so opening a file cannot execute code (unlike
`torch.load` on `.bin`/`.pt` files, which our loader refuses). Tensors can be
memory-mapped and read one at a time. We can parse the header with nothing but
`struct` and `json`:

In [ ]:
path = ckpt_dir / "model.safetensors"
with path.open("rb") as f:
    n = struct.unpack("<Q", f.read(8))[0]
    header = json.loads(f.read(n))
meta = header.pop("__metadata__", None)
print(f"header: {n:,} bytes, {len(header)} tensors, metadata {meta}")
print(f"file size: {path.stat().st_size / 2**20:,.1f} MiB")
for name in list(header)[:5]:
    print(f"  {name:45s} {header[name]['dtype']:5s} {header[name]['shape']}")
print("  ...")
print("dtypes in file:", {v["dtype"] for v in header.values()})
print("lm_head.weight stored:", "lm_head.weight" in header)

## 2. Mapping checkpoint names to our modules

We named our modules exactly like the official checkpoint, but the loader still
builds the map **from the config** (`build_name_map`) and checks every shape
(`expected_shapes`). A renamed or missing module therefore fails loudly.

| checkpoint tensor | shape (0.6B) | our module |
|---|---|---|
| `model.embed_tokens.weight` | [151936, 1024] | `nn.Embedding` (also the LM head) |
| `model.layers.i.self_attn.q_proj.weight` | [2048, 1024] | `nn.Linear(H, Nq·D)` |
| `model.layers.i.self_attn.k_proj.weight` | [1024, 1024] | `nn.Linear(H, Nkv·D)` |
| `model.layers.i.self_attn.q_norm.weight` | [128] | per-head RMSNorm over D |
| `model.layers.i.mlp.gate_proj.weight` | [3072, 1024] | SwiGLU gate |
| `model.norm.weight` | [1024] | final RMSNorm |

`nn.Linear` stores its weight as `[out_features, in_features]`, which is the same
layout as the checkpoint, so no transpose is needed (a classic bug when porting to
other frameworks, and something to remember for the C++ port).

In [ ]:
full = expected_shapes(QWEN3_0_6B_BASE)
print(
    f"Qwen3-0.6B: {len(build_name_map(QWEN3_0_6B_BASE))} tensors, "
    f"{sum(torch.Size(s).numel() for s in full.values()):,} values (tied: no lm_head)"
)
for k in [
    "model.layers.0.self_attn.q_proj.weight",
    "model.layers.0.self_attn.k_proj.weight",
    "model.layers.0.self_attn.o_proj.weight",
    "model.layers.0.mlp.down_proj.weight",
]:
    print(f"  {k:42s} {full[k]}")
ckpt = SafetensorsCheckpoint(ckpt_dir)
first_name = next(iter(ckpt))
print(f"\nthis checkpoint: {len(ckpt)} tensors; first: {first_name} {ckpt.shape_of(first_name)}")

## 3. BF16 vs FP32

The checkpoint is stored in **bfloat16**: 1 sign bit, 8 exponent bits (the same range
as float32), 7 mantissa bits (about 2–3 significant decimal digits). Converting bf16 to
fp32 is **exact** (it only appends zero bits), so a float32 model holds exactly the
trained values; the reverse direction rounds.

We load in float32 for correctness checks. bf16 halves the memory, but on CPUs
without native bf16 instructions (like the Ryzen 3 3250U used here) it is **not**
automatically faster: the kernels may convert internally.

In [ ]:
w = ckpt["model.layers.0.mlp.up_proj.weight"]
w32 = w.to(torch.float32)
print(
    "stored dtype:",
    w.dtype,
    "| bf16 -> fp32 -> bf16 exact:",
    torch.equal(w32.to(torch.bfloat16), w),
)
x = torch.tensor([1.0, 1.0 + 2**-8, 1.0 + 2**-7, 3.14159265])
print("fp32 values      :", x.tolist())
print("rounded to bf16  :", x.to(torch.bfloat16).float().tolist())
print(
    "bf16 eps = 2**-7 =",
    torch.finfo(torch.bfloat16).eps,
    "| fp32 eps =",
    torch.finfo(torch.float32).eps,
)

## 4–5. Checkpoint vs random initialisation; loading

`Qwen3ForCausalLM(config)` starts from random numbers (`N(0, 0.02)`), which is fine
for testing shapes and numerics but means the model knows nothing. Loading replaces
**every** parameter with a trained value. `load_pretrained`:

1. builds the model on the `meta` device (shapes only, no memory, no random init);
2. allocates uninitialised storage in the target dtype (`to_empty`);
3. rebuilds the RoPE buffers and re-ties `lm_head` to `embed_tokens`;
4. copies the checkpoint tensors one at a time with an explicit dtype conversion;
5. fails if anything is missing, unexpected, wrongly shaped, or left unwritten.

In [ ]:
model, report = load_pretrained(ckpt_dir, dtype=torch.float32)
print(report.summary())
print("all mapped:", report.ok, "| parameters:", f"{report.num_parameters:,}")

## 6. Tied embeddings

Qwen3-0.6B ties the LM head to the token embedding: logits are the dot products of
the final hidden state with every token's embedding row. The checkpoint therefore
does **not** contain `lm_head.weight` (596,049,920 stored values = the tied
parameter count). After loading, both names refer to one tensor:

In [ ]:
print(
    "lm_head.weight is embed_tokens.weight:",
    model.lm_head.weight is model.model.embed_tokens.weight,
)
h = torch.randn(1, model.config.hidden_size)
print("logits == h @ E^T:", torch.allclose(model.lm_head(h), h @ model.model.embed_tokens.weight.T))

## 7. Tokenising a real prompt

The tokenizer turns text into ids and back. We use the checkpoint's own
`tokenizer.json` (byte-level BPE) through `Qwen3Tokenizer`, a thin adapter kept
separate from the model. Qwen3 adds **no BOS token**; `<|endoftext|>` (151643) is
both EOS and PAD. Text is NFC-normalised, which matters for Vietnamese: a decomposed
"ệ" (e + two combining marks) becomes one code point.

In [ ]:
tok = Qwen3Tokenizer.from_pretrained(ckpt_dir)
print(tok)
for text in ["Artificial intelligence is", "Hà Nội là thủ đô của"]:
    ids = tok.encode(text)
    print(f"{text!r} -> {len(ids)} ids {ids}")
    print("   pieces:", [tok.decode([i]) for i in ids])
    print("   round trip ok:", tok.decode(ids) == text)

## 8. Verifying logits

The reference is Hugging Face's `Qwen3ForCausalLM` on the same files (eager
attention, eval mode). Both run the same operations, so in float32 on the same CPU
we expect agreement to rounding level; the full report (every layer, five prompts,
greedy ids) comes from `scripts/verify_pretrained.py`. Here, if `transformers` is
installed, we compare the logits directly.

In [ ]:
prompt_ids = torch.tensor([tok.encode("The capital of France is")])
with torch.inference_mode():
    ours = model(prompt_ids).logits
try:
    from transformers.models.qwen3.modeling_qwen3 import Qwen3ForCausalLM as HFQwen3

    hf = HFQwen3.from_pretrained(ckpt_dir, dtype=torch.float32, attn_implementation="eager").eval()
    with torch.inference_mode():
        ref = hf(prompt_ids).logits
    diff = (ours - ref).abs()
    print(
        f"max |ours - HF| = {diff.max():.3e}, relative L2 = {(ours - ref).norm() / ref.norm():.3e}"
    )
    del hf
except ImportError:
    print("transformers not installed: skipping the direct comparison")

## 9. Next-token probabilities

In [ ]:
probs = torch.softmax(ours[0, -1].float(), dim=-1)
top = torch.topk(probs, 8)
print("most likely next tokens after the prompt:")
for p, i in zip(top.values.tolist(), top.indices.tolist(), strict=True):
    piece = tok.decode([i], skip_out_of_vocab=True)
    print(f"  {i:7d} {piece!r:14s} p = {p:.4f}")
if not REAL:
    print("(random weights: these probabilities mean nothing)")

## 10. A short completion

Qwen3-0.6B-**Base** is a pre-trained model, not a chat assistant: it *continues*
text. Prompts should be the beginning of a document. Greedy decoding below uses our
own loop (full recomputation each step; the KV cache follows in M2.2).

In [ ]:
out = greedy_generate(model, prompt_ids, 12, eos_token_id=tok.eos_token_id)
print(repr(tok.decode(out[0], skip_out_of_vocab=True)))

## 11. Diagnosing a wrong mapping

What does a mapping bug look like? We deliberately swap the `gate_proj` and `up_proj`
weights of one layer (same shape, so a shape check cannot catch it) and compare every
layer's output with the correct model. The first layer whose output changes points at
the bug. `scripts/verify_pretrained.py` reports exactly this "first divergence".

In [ ]:
mlp = model.model.layers[min(1, model.config.num_hidden_layers - 1)].mlp


def swap_gate_up():  # in place (no second copy of a 0.6B model in memory)
    with torch.no_grad():
        g = mlp.gate_proj.weight.clone()
        mlp.gate_proj.weight.copy_(mlp.up_proj.weight)
        mlp.up_proj.weight.copy_(g)


with torch.inference_mode():
    good = model(prompt_ids, output_hidden_states=True).hidden_states
swap_gate_up()
with torch.inference_mode():
    bad = model(prompt_ids, output_hidden_states=True).hidden_states
swap_gate_up()  # restore the correct weights

rels = [float((a - b).norm() / a.norm()) for a, b in zip(good, bad, strict=True)]
first = next(i for i, r in enumerate(rels) if r > 1e-6)
for i, rel in enumerate(rels):
    name = "embeddings" if i == 0 else f"after layer {i - 1}"
    print(
        f"{name:16s} relative difference {rel:.2e}"
        + ("   <-- first divergence" if i == first else "")
    )

Other typical mistakes and their signatures:

* **missing tensor silently left random**: impossible here (strict loading and the
  "parameters left uninitialised" check);
* **K and V swapped** or **q_norm/k_norm swapped**: divergence at layer 0;
* **untied head when the config ties it**: correct hidden states but wrong logits;
* **wrong `rope_theta`** (10,000 instead of 1,000,000): small differences at layer 0
  that grow with position; the first token (position 0, no rotation) still matches;
* **bf16 model compared with an fp32 reference using fp32 tolerances**: a "failure" that
  is only rounding; use the documented per-dtype tolerances.

## Exercises

1. Which tensors would a shape check alone *not* distinguish? (Hint: q/k/v in a model
   with `Nq == Nkv`, gate/up.)
2. Compute the checkpoint size in bf16 from the config and compare with the file size.
3. Load the model in bfloat16 and measure the logit difference to float32.